# Fed-XRay: Free Cloud GPU Runner (Google Colab / Kaggle)

Bu notebook, **Fed-XRay** projesinin FastAPI ve PyTorch arka plan sunucusunu **ücretsiz bulut GPU (NVIDIA T4 / 16-32 GB VRAM + 13-30 GB RAM)** üzerinde çalıştırır ve **Cloudflare Tunnel** ile güvenli genel bir HTTPS bağlantı adresi üretir.

### Adımlar:
1. **Menüden GPU Seçin:** `Runtime` -> `Change runtime type` -> `T4 GPU` seçin ve `Save` deyin.
2. **Tümünü Çalıştırın:** `Runtime` -> `Run all` (veya `Ctrl + F9`).
3. En alttaki hücrede çıkan `https://xxxx.trycloudflare.com` adresini kopyalayıp yerel frontend'inizde kullanın.

In [ ]:
# 1. GPU ve Donanım Bilgisini Kontrol Edin
!nvidia-smi

In [ ]:
# 2. Depoyu Klonlayın ve Bağımlılıkları Yükleyin
import os
if not os.path.exists("Fed-XRay"):
    !git clone https://github.com/yusufcalisir/Fed-XRay.git
%cd Fed-XRay
!git pull origin main
!pip install -r requirements.txt

In [ ]:
# 3. Cloudflare Tünelini ve FastAPI Sunucusunu Başlatın
import subprocess
import time
import re
import os

# Cloudflare Tunnel indir
if not os.path.exists("/usr/local/bin/cloudflared"):
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

# Uvicorn backend başlat
print("Starting Fed-XRay FastAPI backend on port 8000...")
backend_proc = subprocess.Popen(
    ["uvicorn", "api.main:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)
time.sleep(3)

# Cloudflare Tüneli başlat
print("Launching Cloudflare Secure Tunnel...")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
for _ in range(40):
    line = tunnel_proc.stdout.readline()
    if line:
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            public_url = match.group(0)
            break
    time.sleep(0.5)

if public_url:
    print("\n" + "="*70)
    print(" FED-XRAY BULUT GPU SUNUCUSU AKTIF!")
    print(f" BAGLANTI ADRESINIZ (Public API URL): {public_url}")
    print("="*70)
    print("\nBu adresi kopyalayip bilgisayarinizdaki frontend/.env.local dosyasina ekleyin:")
    print(f"NEXT_PUBLIC_API_URL={public_url}")
else:
    print("Tunel baglantisi saglanamadi. Lutfen loglari kontrol edin.")

In [ ]:
# 4. Canlı Sunucu Loglarını Takip Edin (Oturumu Açık Tutar)
try:
    while True:
        line = backend_proc.stdout.readline()
        if line:
            print(line.strip())
        time.sleep(0.1)
except KeyboardInterrupt:
    print("Sunucu kapatiliyor...")
    backend_proc.terminate()
    tunnel_proc.terminate()